# GUDS-EDL Comprehensive Ablation Sweep
This notebook orchestrates the massive ablation study defined in Phase 5.
It iterates through Structural Baselines, Component Ablations, and Mathematical Ablations across 3 random seeds.

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# Define seeds for statistical robustness
SEEDS = [42, 100, 2024]

# Define the comprehensive ablation configurations
ablation_configs = [
    # --- 1. Structural Baselines ---
    {'group': 'Structural Baselines', 'name': 'Dense EDL (Upper Bound)', 'flags': '--model dense_edl --sparsity 0.0'},
    {'group': 'Structural Baselines', 'name': 'Static 2:4 Sparse EDL', 'flags': '--model guds_edl --static_24_mask True'},
    {'group': 'Structural Baselines', 'name': 'RigL-style Dynamic 2:4', 'flags': '--model guds_edl --pruner magnitude --regrower gradient'},
    {'group': 'Structural Baselines', 'name': 'Random Dynamic 2:4', 'flags': '--model guds_edl --pruner random --regrower random'},
    
    # --- 2. Component Ablations ---
    {'group': 'Component Ablations', 'name': 'w/o Uncertainty Pruner', 'flags': '--model guds_edl --disable_pruner True'},
    {'group': 'Component Ablations', 'name': 'w/o Evidence Regrower', 'flags': '--model guds_edl --disable_regrower True'},
    {'group': 'Component Ablations', 'name': 'w/o Topology Cache', 'flags': '--model guds_edl --use_topology_cache False'},
    {'group': 'Component Ablations', 'name': 'w/o Anti-Crystallization', 'flags': '--model guds_edl --use_anticryst False'},
    {'group': 'Component Ablations', 'name': 'w/o Asymmetric KL', 'flags': '--model guds_edl --kl_scaling symmetric'},
    {'group': 'Component Ablations', 'name': 'w/o Evidential Focal Loss', 'flags': '--model guds_edl --loss standard_edl'},
    {'group': 'Component Ablations', 'name': 'w/o Bias-Corrected TS', 'flags': '--model guds_edl --calibration standard_ts'},

    # --- 3. Mathematical Ablations ---
    {'group': 'Mathematical Ablations', 'name': 'Absolute Pruner (Magnitude)', 'flags': '--model guds_edl --pruner absolute_grad'},
    {'group': 'Mathematical Ablations', 'name': 'KL-to-Uniform Regrower', 'flags': '--model guds_edl --regrower kl_uniform'},

    # --- 4. Proposed Framework ---
    {'group': 'Proposed', 'name': 'Full GUDS-EDL', 'flags': '--model guds_edl'}
]

results_db = []

def run_experiment(config, seed):
    print(f"\n{'='*50}")
    print(f"Running: {config['name']} | Seed: {seed}")
    print(f"Flags: {config['flags']}")
    print(f"{'='*50}")
    
    # Construct your training command here.
    # Example: cmd = f"python train.py --dataset isic2024 --seed {seed} {config['flags']}"
    # subprocess.run(cmd, shell=True)
    
    # Simulated Metric Collection (Replace with actual JSON/CSV parsing from the run output)
    metrics = {
        'group': config['group'],
        'name': config['name'],
        'seed': seed,
        'Macro-AUROC': np.random.uniform(0.85, 0.92),
        'pAUC_0.80': np.random.uniform(0.09, 0.14),
        'ECE': np.random.uniform(0.06, 0.15),
        'AURC': np.random.uniform(0.0001, 0.0004),
        'Spec_at_80Sens': np.random.uniform(0.20, 0.95),
        'Mask Turnover Rate': np.random.uniform(0.0, 0.15),
        'Dead Channel Ratio': np.random.uniform(0.0, 0.05)
    }
    return metrics

# Execute the sweep
for config in ablation_configs:
    for seed in SEEDS:
        res = run_experiment(config, seed)
        results_db.append(res)

# Process results to compute Mean ± Std
df = pd.DataFrame(results_db)
agg_df = df.groupby(['group', 'name']).agg(['mean', 'std']).reset_index()

print("\n\nSweep Complete! Ready to export to LaTeX.")
display(agg_df)
